# DeepClean
> Data Cleaning Tool

**This tool handles:**
* Missing Values
* Outliers
* Categorical Encoding
* Date Features
* Normalization / Standardization
* Duplicates

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta
import random
from scipy import stats
import gc

print("Libraries imported successfully!")

Libraries imported successfully!


## Data Generation Function

Uses vectorized numpy operations to ensure it can scale to millions of rows quickly without relying on slow pandas `.apply()` methods during the base generation.

In [2]:
def generate_user_data(n_rows=1_000_000):
    print(f"Generating {n_rows} base rows for user schema...")
    
    np.random.seed(42)
    
    # 1. Age
    age = np.random.normal(loc=35.0, scale=12.0, size=n_rows)
    age = np.clip(age, 18, 90).astype(int)
    
    # 2. Income
    income = np.random.lognormal(mean=10.5, sigma=0.8, size=n_rows)
    income = np.round(income, 2)
    
    # 3. City
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio']
    city_probs = [0.25, 0.20, 0.15, 0.15, 0.10, 0.10, 0.05]
    city_data = np.random.choice(cities, size=n_rows, p=city_probs)
    
    # 4. Joined Date
    base_date = np.datetime64('2015-01-01')
    random_days = np.random.randint(0, 3000, size=n_rows)
    joined_date = base_date + random_days.astype('timedelta64[D]')
    
    # 5. Notes
    phrase_pool = [
        "Customer complained about service.",
        "Requested a refund.",
        "VIP account upgraded.",
        "Left a 5-star review.",
        "Inactive for 6 months.",
        "Password reset required.",
        "Pending address verification.",
        "No issues reported."
    ]
    notes = np.random.choice(phrase_pool, size=n_rows)
    
    df = pd.DataFrame({
        'age': age,
        'income': income,
        'city': city_data,
        'joined_date': joined_date,
        'notes': notes
    })

    print("Injecting anomalies...")

    # --- MISSING VALUES ---
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.08), replace=False), 'age'] = np.nan
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.12), replace=False), 'income'] = np.nan
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.50), replace=False), 'notes'] = np.nan

    # --- OUTLIERS ---
    age_outlier_idx = np.random.choice(df.index, size=int(n_rows * 0.005), replace=False)
    df.loc[age_outlier_idx, 'age'] = np.random.choice([-15, -5, 150, 999], size=len(age_outlier_idx))
    
    inc_outlier_idx = np.random.choice(df.index, size=int(n_rows * 0.005), replace=False)
    df.loc[inc_outlier_idx, 'income'] = df.loc[inc_outlier_idx, 'income'] * np.random.uniform(50, 500, size=len(inc_outlier_idx))
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.001), replace=False), 'income'] = -5000.00

    # --- MESSY CATEGORICALS ---
    messy_city_idx = np.random.choice(df.dropna(subset=['city']).index, size=int(n_rows * 0.07), replace=False)
    df.loc[messy_city_idx, 'city'] = df.loc[messy_city_idx, 'city'].apply(
        lambda x: f"  {str(x).lower()} " if np.random.rand() > 0.5 else f"{str(x).upper()}   "
    )

    # --- DUPLICATES ---
    duplicates = df.sample(frac=0.03, random_state=42)
    df = pd.concat([df, duplicates], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Generation complete. Final shape: {df.shape}")
    return df

## Execution and Memory Management

**Note on Memory Constraints:** Adjust `n_rows` carefully. If your machine freezes, it is likely due to the Jupyter kernel retaining data copies in memory. Use `gc.collect()` if doing multiple runs.

In [3]:
# Force garbage collection to clear ghost references before generation
gc.collect() 

# WARNING: Keep n_rows low (e.g., 10,000) for logic testing in notebooks. 
df = generate_user_data(n_rows=10_000)

In [4]:
df.head()

## Verify Targets

In [ ]:
print("--- Data Info ---")
df.info()

print("\n--- Missing Values ---")
display(df.isnull().sum())

print("\n--- Outlier Check (Age) ---")
display(df['age'].describe())

print("\n--- Outlier Check (Income) ---")
display(df['income'].describe())

## Tool Definition

In [ ]:
class DeepClean:
    def __init__(self, df):
        """Initialize the cleaning tool with a copy of the dataframe."""
        self.df = df.copy()
        self.report = {}

    def remove_duplicates(self):
        """Removes exact duplicate rows from the dataframe."""
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        self.report['duplicates_removed'] = before - len(self.df)
        return self.df

    def handle_missing_values(self, strategy='mean', columns=None):
        """Fills missing values in specified columns or all numeric columns."""
        cols_to_fix = columns if columns else self.df.select_dtypes(include=[np.number]).columns
        missing_before = self.df[cols_to_fix].isnull().sum().sum()
        
        imputer = SimpleImputer(strategy=strategy)
        self.df[cols_to_fix] = imputer.fit_transform(self.df[cols_to_fix])
        
        self.report['missing_values_filled'] = int(missing_before)
        return self.df

    def handle_outliers(self, threshold=3.0):
        """Clips outliers using the Z-score method for all numeric columns."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        outliers_count = 0
        
        for col in numeric_cols:
            z_scores = np.abs(stats.zscore(self.df[col].dropna()))
            outlier_mask = z_scores > threshold
            outliers_count += outlier_mask.sum()
            
            # Clip values to the threshold boundaries
            upper = self.df[col].mean() + threshold * self.df[col].std()
            lower = self.df[col].mean() - threshold * self.df[col].std()
            self.df[col] = self.df[col].clip(lower=lower, upper=upper)
            
        self.report['outliers_clipped'] = int(outliers_count)
        return self.df

    def process_dates(self, date_columns=None):
        """Converts columns to datetime and extracts features (year, month, day, dayofweek)."""
        cols = date_columns if date_columns else self.df.select_dtypes(include=['datetime64', 'object']).columns
        processed_count = 0
        
        for col in cols:
            try:
                # Convert to datetime if it isn't already
                if not pd.api.types.is_datetime64_any_dtype(self.df[col]):
                    self.df[col] = pd.to_datetime(self.df[col], errors='coerce')
                
                if pd.api.types.is_datetime64_any_dtype(self.df[col]):
                    # Extract features
                    self.df[f'{col}_year'] = self.df[col].dt.year
                    self.df[f'{col}_month'] = self.df[col].dt.month
                    self.df[f'{col}_day'] = self.df[col].dt.day
                    self.df[f'{col}_dayofweek'] = self.df[col].dt.dayofweek
                    processed_count += 1
            except Exception:
                continue
                
        self.report['date_columns_processed'] = processed_count
        return self.df

    def normalize_data(self, method='standard'):
        """Scales numerical data using StandardScaler or MinMaxScaler."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        scaler = StandardScaler() if method == 'standard' else MinMaxScaler()
        
        self.df[numeric_cols] = scaler.fit_transform(self.df[numeric_cols])
        self.report['normalization_method'] = method
        return self.df

    def encode_categorical(self):
        """Simplistic Label Encoding for object-type columns."""
        obj_cols = self.df.select_dtypes(include=['object']).columns
        le = LabelEncoder()
        
        for col in obj_cols:
            self.df[col] = le.fit_transform(self.df[col].astype(str))
            
        self.report['encoded_columns_count'] = len(obj_cols)
        return self.df

    def get_summary(self):
        """Returns a summary of all cleaning operations performed."""
        return self.report